# Python foundations — beginner → advanced (AI engineering track)

**How to use this notebook**

1. Run cells **top to bottom** (each section builds on earlier ideas).
2. Read the markdown **Explanation** blocks before running **Examples**.
3. Modify values and break things — that is how they stick.

**Requires:** Python 3.10+ recommended (uses `|` union syntax in places). Standard library only except optional comments.

---

## Part 1 — Beginner core

### Variables, types, and operators

**Explanation:** Python figures out types at runtime (`dynamic typing`). You still think in types: numbers for math, strings for text, booleans for conditions. AI pipelines almost always move between **strings** (prompts), **structured dicts/list** (JSON-like tool payloads), and **numbers** (embeddings).

**Examples:**

In [ ]:
# Numbers and strings
batch_size = 8
model_name = "gpt-demo"
temperature = 0.7

# f-strings — you'll use these constantly for prompts
user_prompt = "summarize"
full_prompt = f"Task: {user_prompt}\nBatch size: {batch_size}"
print(full_prompt)

# Booleans and comparisons
use_streaming = True
print(use_streaming and batch_size > 0)

### Lists, tuples, dicts, sets

**Explanation:**
- **list** — ordered, mutable — chunks of text, batches of IDs.
- **tuple** — ordered, immutable — often small fixed records `(role, content)`.
- **dict** — key → value — JSON-shaped API payloads; extremely common.
- **set** — unique elements — dedupe URLs / doc IDs quickly.

**Examples:**

In [ ]:
messages = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Hello!"},
]

# Safe access patterns
first_user_text = messages[-1]["content"] if messages else ""
print("Last message:", first_user_text)

# Dict merge (Python 3.9+)
defaults = {"temperature": 0.7, "max_tokens": 256}
overrides = {"temperature": 0.2}
params = defaults | overrides
print(params)

# Unique chunk ids
chunk_ids = ["a", "b", "a", "c"]
print(set(chunk_ids))

### Control flow — `if`, loops

**Explanation:** Branch on errors, empty retrieval results, flags from configs. **`enumerate`** gives index + item — handy when chunking lists.

**Examples:**

In [ ]:
chunks = ["alpha", "beta", "gamma"]

for i, text in enumerate(chunks):
    if len(text) < 4:
        print(i, "short chunk", text)
    else:
        print(i, "ok", text)

### Functions — arguments and returns

**Explanation:** `*args` collects extra positional arguments; `**kwargs` collects keyword arguments. APIs often forward unknown keys → `kwargs` pattern appears in SDK wrappers.

**Examples:**

In [ ]:
def build_messages(system: str, user: str, **extra_fields: str) -> list[dict[str, str]]:
    """Tiny messages builder — extra_fields might hold optional metadata strings."""
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    if extra_fields:
        msgs.append({"role": "system", "content": repr(extra_fields)})
    return msgs


print(build_messages("Be concise.", "Hi", locale="en"))

---
## Part 2 — Intermediate patterns

### Classes and instances

**Explanation:** Bundle state + behavior — tool clients, embedders, chunkers. Prefer **explicit constructors** (`__init__`) over globals.

**Examples:**

In [ ]:
class SimpleEmbedder:
    """Dummy embedder — replaces API with fixed-width fake vectors."""

    def __init__(self, dim: int = 4) -> None:
        self.dim = dim

    def embed(self, texts: list[str]) -> list[list[float]]:
        return [[float(len(t) % self.dim)] * self.dim for t in texts]


emb = SimpleEmbedder(dim=3)
vectors = emb.embed(["hello", "world"])
print(vectors)

### Exceptions — failures are normal in AI systems

**Explanation:** Network timeouts, rate limits, malformed JSON from models — handle **narrowly** (specific exception types where possible), log context, optionally retry at a higher layer.

**Examples:**

In [ ]:
import json


def parse_tool_json(raw: str) -> dict:
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"invalid tool JSON: {e}") from e


print(parse_tool_json('{"tool": "search", "query": "rag"}'))

try:
    parse_tool_json("not json")
except ValueError as err:
    print("Caught:", err)

### Comprehensions — concise transforms

**Explanation:** Same as loops but often clearer for filtering/mapping lists — preprocessing texts before embedding.

**Examples:**

In [ ]:
raw_docs = [" Hello ", "", "WORLD", "  rag  "]

normalized = [s.strip().lower() for s in raw_docs if s.strip()]
print(normalized)

### Files & JSON — ingestion starter kit

**Explanation:** RAG starts with reading sources — JSONL is common (`one JSON object per line`).

**Examples:**

In [ ]:
import json
from pathlib import Path

demo_dir = Path("_notebook_tmp")
demo_dir.mkdir(exist_ok=True)
sample_file = demo_dir / "sample.jsonl"

sample_file.write_text(
    '{"id": 1, "text": "hello"}\n{"id": 2, "text": "world"}\n',
    encoding="utf-8",
)

records = []
for line in sample_file.read_text(encoding="utf-8").splitlines():
    records.append(json.loads(line))

print(records)

# cleanup — optional
sample_file.unlink(missing_ok=True)
demo_dir.rmdir()

---
## Part 3 — Advanced (high value for AI engineers)

### Type hints — contracts without ceremony

**Explanation:** Annotate public functions so teammates + IDEs catch mistakes early. AI codebases glue many libraries — types reduce chaos.

**Examples:**

In [ ]:
from typing import Optional


def cosine_similarity_naive(a: list[float], b: list[float]) -> float:
    if len(a) != len(b):
        raise ValueError("dimension mismatch")
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(y * y for y in b) ** 0.5
    return dot / (na * nb) if na and nb else 0.0


def nearest_neighbor(
    query: list[float],
    corpus: list[list[float]],
    labels: Optional[list[str]] = None,
) -> tuple[int, float]:
    best_i, best_sim = max(
        ((i, cosine_similarity_naive(query, vec)) for i, vec in enumerate(corpus)),
        key=lambda t: t[1],
    )
    return best_i, best_sim


q = [1.0, 0.0]
corpus = [[0.9, 0.1], [0.0, 1.0]]
idx, sim = nearest_neighbor(q, corpus)
print("best idx", idx, "similarity", round(sim, 4))

### `dataclass` — configs and immutable-ish records

**Explanation:** Less boilerplate than manual `__init__`. Use **`frozen=True`** for immutable configs; **`slots=True`** saves memory at scale.

**Examples:**

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True, slots=True)
class RetrievalConfig:
    top_k: int = 5
    score_threshold: float = 0.2


cfg = RetrievalConfig(top_k=3)
print(cfg)

# Frozen prevents accidental mutation — mirrors "good API discipline"
try:
    cfg.top_k = 99
except Exception as e:
    print("Expected error:", type(e).__name__)

### Decorators — cross-cutting behavior

**Explanation:** Wrap functions to add timing, retries, logging — keep core logic clean.

**Examples:**

In [ ]:
import time
import functools


def timed(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        dt = time.perf_counter() - t0
        print(f"{fn.__name__} took {dt*1000:.2f} ms")
        return out

    return wrapper


@timed
def fake_embedding_batch(texts: list[str]) -> list[list[float]]:
    time.sleep(0.05)
    return [[float(len(t))] for t in texts]


fake_embedding_batch(["hello", "rag"])

### Generators — memory-friendly pipelines

**Explanation:** **Yield** items one-by-one instead of building giant lists — critical when iterating millions of chunks.

**Examples:**

In [ ]:
from typing import Iterator


def overlapping_windows(text: str, window: int, step: int) -> Iterator[str]:
    """Simple sliding windows — naive but illustrates generators."""
    i = 0
    while i + window <= len(text):
        yield text[i : i + window]
        i += step


text = "abcdefghij"
print(list(overlapping_windows(text, window=4, step=3)))

### Context managers — predictable cleanup

**Explanation:** Acquire/release resources (files, locks, timers) safely — FastAPI dependency injection echoes this idea.

**Examples:**

In [ ]:
import time
from contextlib import contextmanager


@contextmanager
def timer_label(label: str):
    t0 = time.perf_counter()
    yield
    print(f"[{label}] {1000*(time.perf_counter()-t0):.2f} ms")


with timer_label("fake_retrieval"):
    time.sleep(0.03)

### `Protocol` — structural interfaces

**Explanation:** Anything with matching methods satisfies the protocol — ideal for `Embedder`, `VectorStore`, `Tool` abstractions.

**Examples:**

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Embedder(Protocol):
    def embed(self, texts: list[str]) -> list[list[float]]: ...


class MiniEmbedder:
    def embed(self, texts: list[str]) -> list[list[float]]:
        return [[1.0, 0.0] for _ in texts]


client: Embedder = MiniEmbedder()
print(isinstance(client, Embedder))
print(client.embed(["hi"]))

### Async basics — concurrency without threads

**Explanation:** Many AI workloads are **I/O bound** (HTTP to models/databases). `asyncio` lets you overlap waits — foundation before FastAPI async routes.

**Examples:** (`asyncio.run` works from scripts and notebooks in modern Python)

In [ ]:
import asyncio


async def fake_remote_call(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"result-{name}"


async def main():
    # Run multiple I/O-ish tasks concurrently
    results = await asyncio.gather(
        fake_remote_call("a", 0.2),
        fake_remote_call("b", 0.2),
        fake_remote_call("c", 0.2),
    )
    print(results)


await main()

> **Notebook note:** If `await main()` errors (older Jupyter / IPython), replace the last line with:
> ```python
> asyncio.run(main())
> ```
> Modern Jupyter often supports top-level `await`.

---
## Next steps in this repo

- Implement drills from **`CURRICULUM-A-Z.md`** under `exercises/`.
- Move to **`04-api-engineering/`** + **`05-fastapi-projects/`** when HTTP services click.
- Log sessions in **`01-daily-log/`**.

**Suggested libraries later (outside stdlib):** `httpx`, `pydantic`, `numpy`, `pytest` — add them when projects need them.